<a href="https://colab.research.google.com/github/Edenshmuel/CrimeData/blob/main/Baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Required Libraries

In [1]:
import pandas as pd
import zipfile
import requests
from io import BytesIO
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.exceptions import UndefinedMetricWarning

Suppressing Warnings for Cleaner Output

In [2]:
import warnings

# Suppress specific warning categories for a cleaner notebook output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

Loading Supervised Datasets from GitHub Repository

In [3]:
# Define the ZIP file URL
zip_url = "https://github.com/Edenshmuel/CrimeData/raw/main/supervised_dataset.zip"

# Function to load a specific CSV file from the ZIP in GitHub
def load_csv_from_zip(zip_url, inner_file_name):
    response = requests.get(zip_url)
    if response.status_code == 200:
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            with z.open(inner_file_name) as f:
                return pd.read_csv(f)
    else:
        raise Exception("Failed to download supervised_dataset.zip")

# Load datasets from the ZIP
X_train = load_csv_from_zip(zip_url, "X_train_supervised.csv")
X_test = load_csv_from_zip(zip_url, "X_test_supervised.csv")
y_train = load_csv_from_zip(zip_url, "y_train_supervised.csv").values.ravel()
y_test = load_csv_from_zip(zip_url, "y_test_supervised.csv").values.ravel()

Handling Imbalanced Data Using SMOTE

In [4]:
# Apply SMOTE to balance the training dataset
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

Evaluating a Most Frequent Classifier as a Baseline

In [5]:
# Initialize and train a dummy classifier using the most frequent strategy
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train_balanced, y_train_balanced)

# Predict on the test set
y_pred_dummy = dummy_clf.predict(X_test)

# Evaluate the dummy classifier using common metrics
dummy_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_dummy),
    "Precision": precision_score(y_test, y_pred_dummy, average='weighted', zero_division=0),
    "Recall": recall_score(y_test, y_pred_dummy, average='weighted', zero_division=0),
    "F1 Score": f1_score(y_test, y_pred_dummy, average='weighted', zero_division=0),
    "Confusion Matrix": confusion_matrix(y_test, y_pred_dummy)
}

# Print metrics and classification report
print("Most Frequent Classifier Metrics:")
for metric, value in dummy_metrics.items():
    print(f"{metric}: {value}")

# Display a detailed classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred_dummy, zero_division=0))

Most Frequent Classifier Metrics:
Accuracy: 0.0010796074961240119
Precision: 1.1655523456871584e-06
Recall: 0.0010796074961240119
F1 Score: 2.328590727369644e-06
Confusion Matrix: [[   438      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [  7149      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [  4253      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [ 23599      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [139327      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [  7756      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [   104      0      0      0      0      0      0      0      0      0
       0      0      0      0]
 [ 19803      0      0      0      0      0      0      0      0      0
       0      0      0     

The DummyClassifier baseline achieved an accuracy of approximately 0.1%, indicating that predicting only the most frequent class performs extremely poorly for this multi-class classification task. This result provides a minimal benchmark for evaluating the performance of the trained models.